# Experimento diário: dos dados aos CSVs

Este notebook apresenta o fluxo novo. As células de execução são explícitas: rode somente a etapa desejada. O treinamento usa a grade científica completa e pode demorar. As saídas históricas pertencem ao protocolo anterior; este notebook não contém resultados científicos novos.


## 1. Configuração e moedas

Os parâmetros vêm dos dois JSONs. Escolha um identificador novo para uma configuração diferente. Aqui usamos somente BTCUSDT; o mesmo fluxo aceita mais moedas.


In [ ]:
from IPython.display import display
from tcc_crypto.config import load_config, select_symbols
from tcc_crypto.paths import ROOT, Paths
from tcc_crypto import experiment

config = load_config()
symbols = select_symbols(config, ["BTCUSDT"])
paths = Paths(ROOT, "review_01")
display(config.signature, symbols)


## 2. Localização e coleta

Os dados brutos existentes são preservados. A chamada abaixo coleta ou retoma apenas as moedas escolhidas; pule-a para usar dados já salvos. Mostrar caminhos não cria arquivos.


In [ ]:
display(paths.processed, paths.output)


In [ ]:
experiment.collect(config, symbols)


## 3. Disponibilidade e tratamento

A preparação valida a vela inteira antes das features e dos alvos. Lacunas internas recebem os valores da última vela válida, com a origem registrada. A disponibilidade continua refletindo os dados brutos. Reexecutar uma etapa exige `overwrite=True`; os resultados posteriores devem ser reconstruídos quando suas entradas mudarem.


In [ ]:
experiment.prepare(config, symbols, paths)


In [ ]:
display(experiment.inspect(config, paths, "BTCUSDT", "availability"))
display(experiment.inspect(config, paths, "BTCUSDT", "treatment"))


## 4. Features e alvos

Cada linha é o dia das features. O alvo é o resultado do dia seguinte, para 1%, 2% e 3%. O fim sem alvo futuro continua ausente. As primeiras linhas podem precisar de aquecimento. O Parabolic SAR mantém a ressalva científica já documentada.


In [ ]:
experiment.build_datasets(config, symbols, paths)


In [ ]:
display(experiment.inspect(config, paths, "BTCUSDT", "features"))
display(experiment.inspect(config, paths, "BTCUSDT", "targets"))
display(experiment.inspect(config, paths, "BTCUSDT", "dataset"))


## 5. Janelas externas e divisões internas

O in-sample tem 60, 75 ou 90 dias. O bloco seguinte é validação e depois vira treinamento. Apenas a última validação pode ser parcial. A busca usa cinco divisões temporais dentro de cada treinamento, sem embaralhar. O ajuste ocorre após o fechamento do primeiro dia de features da validação, quando o último alvo de treinamento já é conhecido; a previsão é para o dia seguinte. A consulta abaixo não treina.


In [ ]:
display(experiment.inspect_windows(config, paths, "BTCUSDT"))


## 6. Treinamento e avaliação

A seleção e a normalização são ajustadas dentro de cada divisão da busca. A escolha considera F1 e os desempates aprovados. O procedimento vencedor é reajustado no in-sample completo. Nenhum alvo de validação entra no ajuste. Esta célula executa a grade completa para as moedas selecionadas.


In [ ]:
experiment.train(config, symbols, paths)


## 7. Métricas e seleção

Precision e recall são por bloco e papel. As comparações usam datas válidas comuns. Os filtros registram variância, correlações, escolhas, descartes e eventual alternativa de maior variância. Falhas ficam explícitas e não viram sinais zero.


In [ ]:
display(experiment.inspect(config, paths, "BTCUSDT", "metrics"))
display(experiment.inspect(config, paths, "BTCUSDT", "selection"))
display(experiment.inspect(config, paths, "BTCUSDT", "failures"))


## 8. Consulta das saídas

As matrizes principais estão na pasta da execução. A auditoria detalhada tem os identificadores de bloco de cada moeda. A etapa abaixo apenas reconstrói CSVs a partir dos resultados salvos; não treina novamente. `overwrite=True` autoriza substituir as consultas já geradas.


In [ ]:
experiment.consolidate(config, paths, overwrite=True)
display(paths.main("metrics"), paths.main("feature_selection"))
display(paths.main("predictions_in_sample"), paths.main("actuals_in_sample"))
display(paths.main("predictions_out_of_sample"), paths.main("actuals_out_of_sample"))
